# Session 3: building the flood hydrograph
**A. Real-world problem.** A stream gauge does not show one peak number, it shows a curve of
flow against time: the *hydrograph*. A storm has several bursts, and each burst makes its own
small flood; the river carries the sum. Then a developer paves part of the valley. How much
higher and earlier is the new peak, and how big a pond keeps it at the old value?

**Plan.** (1) excess rain from the curve number, (2) a triangular unit hydrograph (UH),
(3) convolution of burst by burst, (4) Muskingum channel routing, (5) a detention pond by
level-pool routing, (6) the story of urbanisation.

*Educational model.* Real flood design belongs to qualified hydrologists and official agencies.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

DT = 0.5                                                          # time step and burst length (h)
STORM = np.array([4.0, 8.0, 14.0, 26.0, 18.0, 10.0, 6.0, 4.0])    # rain per 0.5 h burst (mm), total 90 mm
AREA = 12.0                                                       # km2, a made-up catchment
CN_PRE, TC_PRE = 70.0, 2.0                                        # before development: curve number, Tc (h)
CN_POST, TC_POST = 90.0, 1.2                                      # after: paved and smoother, so faster

## Step 1: excess rain for each burst
The SCS run-off (Session 2) is applied to the *cumulative* rain, and each burst gets the extra
run-off it adds. Early bursts mostly fill the ground (Ia), later ones run off more.

In [ ]:
def scs_runoff(P, CN):
    """Run-off depth (mm) from rain depth P (mm), curve number CN (S = 25400/CN - 254, Ia = 0.2 S)."""
    S = 25400.0 / CN - 254.0
    x = np.asarray(P, dtype=float) - 0.2 * S
    pos = np.where(x > 0, x, 0.0)
    return np.where(x > 0, pos ** 2 / (pos + S), 0.0)


def excess_rain(pulses, CN):
    """Effective rain per burst (mm): differences of the run-off of the cumulative rain."""
    cum = np.concatenate([[0.0], np.cumsum(pulses)])
    return np.diff(scs_runoff(cum, CN))

## Step 2: the triangular unit hydrograph
The *unit hydrograph* is the river's response to 1 mm of effective rain over the whole
catchment during one burst of length D. The SCS triangular form (standard, approximate):

    Tp = D/2 + 0.6 Tc            time to peak (h)
    Qp = 0.208 * A * Pe / Tp     peak (m3/s), A in km2, Pe = 1 mm, Tp in h
    base time = 2.67 Tp

Check of the 0.208: the area under the triangle is 0.5 * Qp * 2.67 Tp * 3600 s. For that to
equal 1 mm over A km2, i.e. 1000 * A m3, we need Qp = 1000 A / (0.5 * 2.67 * 3600 Tp) =
0.208 A / Tp. The UH keeps the water: its volume is 1 mm over the catchment.

In [ ]:
def triangular_uh(A, Tc, D=DT, dt=DT, normalize=True):
    """Ordinates (m3/s) at times 0, dt, 2dt, ... of the unit hydrograph for 1 mm of effective rain."""
    Tp = D / 2.0 + 0.6 * Tc
    Qp = 0.208 * A / Tp
    tb = 2.67 * Tp
    t = np.arange(0.0, tb + dt, dt)
    u = np.where(t <= Tp, Qp * t / Tp, np.maximum(Qp * (tb - t) / (tb - Tp), 0.0))
    if normalize:                                # make the sampled volume exactly 1 mm over A
        u = u * (1000.0 * A) / (u.sum() * dt * 3600.0)
    return u


def uh_volume_mm(u, A, dt=DT):
    """Depth (mm) of water under the unit hydrograph spread over the catchment (should be 1)."""
    return u.sum() * dt * 3600.0 / (A * 1000.0)

## Step 3: convolution = add up the small floods
Burst k with effective rain Pe[k] produces Pe[k] times the UH, starting at time k*dt. The river
flow at step n is the sum over bursts: Q[n] = sum_k Pe[k] * U[n - k]. That is a
*discrete convolution*; `np.convolve` does exactly this.

In [ ]:
def convolve_manual(pe, u):
    """Q[n] = sum_k pe[k] * u[n-k] written as two plain loops."""
    q = np.zeros(len(pe) + len(u) - 1)
    for k, p in enumerate(pe):
        for j, uj in enumerate(u):
            q[k + j] += p * uj
    return q


def flood(pulses=STORM, CN=CN_PRE, Tc=TC_PRE, A=AREA, dt=DT):
    """Hydrograph of the whole storm. Returns (t in h, Q in m3/s, effective rain per burst)."""
    pe = excess_rain(pulses, CN)
    q = np.convolve(pe, triangular_uh(A, Tc, dt, dt))
    return np.arange(len(q)) * dt, q, pe


def peak_and_time(t, q):
    j = int(np.argmax(q))
    return float(q[j]), float(t[j])

## Step 4: Muskingum channel routing
A flood wave travelling down a river is delayed and flattened. Muskingum routing:

    O[n+1] = C0 I[n+1] + C1 I[n] + C2 O[n]
    C0 = (-K X + dt/2) / D1,  C1 = (K X + dt/2) / D1,  C2 = (K (1-X) - dt/2) / D1,
    D1 = K (1-X) + dt/2         (so C0 + C1 + C2 = 1)

I = inflow, O = outflow (m3/s), K = travel time of the reach (h), X = weighting 0..0.5
(0 = most flattening, 0.5 = pure delay). Keep dt between 2KX and 2K(1-X) so all C are >= 0.

In [ ]:
def muskingum(inflow, K, X, dt=DT):
    """Route an inflow series through a reach. Returns (outflow, (C0, C1, C2))."""
    d1 = K * (1.0 - X) + dt / 2.0
    c0, c1, c2 = (-K * X + dt / 2.0) / d1, (K * X + dt / 2.0) / d1, (K * (1.0 - X) - dt / 2.0) / d1
    out = np.zeros(len(inflow))
    out[0] = inflow[0]
    for n in range(len(inflow) - 1):
        out[n + 1] = c0 * inflow[n + 1] + c1 * inflow[n] + c2 * out[n]
    return out, (c0, c1, c2)

## Step 5: a detention pond (level-pool routing)
A pond with vertical walls of surface area Ap (m2) and an orifice at the bottom.
Water balance: Ap dh/dt = I(t) - O(h). The orifice law (simplified): O = c sqrt(h), with c
chosen so that O = Qmax at the full depth H. Find the smallest Ap that keeps the stage below
H, then the volume is Ap * H. We step with RK4 and interpolate the inflow linearly.

In [ ]:
def route_pond(inflow, dt_h, Ap, H, Qmax, sub=40, t_end_h=None, h0=0.0):
    """Level-pool routing from initial stage h0 (m). Returns (t in h, outflow m3/s, stage in m, stored volume left in m3)."""
    c = Qmax / np.sqrt(H)
    n_in = len(inflow)
    t_end = t_end_h if t_end_h is not None else (n_in - 1) * dt_h + 12.0
    n = int(round(t_end / dt_h))
    I = np.concatenate([inflow, np.zeros(max(n + 2 - n_in, 0))])
    h = h0
    hs, outs = [h0], [c * np.sqrt(h0)]
    hsub = dt_h * 3600.0 / sub

    def rhs(hh, i_now):
        return (i_now - c * np.sqrt(max(hh, 0.0))) / Ap

    for k in range(n):
        for s in range(sub):
            i0 = I[k] + (I[k + 1] - I[k]) * s / sub
            im = I[k] + (I[k + 1] - I[k]) * (s + 0.5) / sub
            i1 = I[k] + (I[k + 1] - I[k]) * (s + 1) / sub
            k1 = rhs(h, i0)
            k2 = rhs(h + 0.5 * hsub * k1, im)
            k3 = rhs(h + 0.5 * hsub * k2, im)
            k4 = rhs(h + hsub * k3, i1)
            h = max(h + hsub * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0, 0.0)
        hs.append(h)
        outs.append(c * np.sqrt(h))
    return np.arange(n + 1) * dt_h, np.array(outs), np.array(hs), Ap * h


def size_pond(inflow, dt_h, Qtarget, H, lo=1.0e3, hi=1.0e7, iters=45):
    """Smallest surface area (m2) for which the stage just reaches H, so the outflow peak is Qtarget."""
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        if route_pond(inflow, dt_h, mid, H, Qtarget)[2].max() > H:
            lo = mid        # pond too small: water would overtop
        else:
            hi = mid
    return hi

## Step 6: run the story

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    t, q0, pe0 = flood(STORM, CN_PRE, TC_PRE)
    t1, q1, pe1 = flood(STORM, CN_POST, TC_POST)
    pk0, tp0 = peak_and_time(t, q0)
    pk1, tp1 = peak_and_time(t1, q1)
    print(f"before: effective rain {pe0.sum():.1f} mm, peak {pk0:.1f} m3/s at {tp0:.1f} h")
    print(f"after : effective rain {pe1.sum():.1f} mm, peak {pk1:.1f} m3/s at {tp1:.1f} h")

    fig, (a1, a2) = plt.subplots(2, 1, figsize=(6.4, 6), sharex=True, gridspec_kw={"height_ratios": [1, 2]})
    tb = np.arange(len(STORM)) * DT
    a1.bar(tb, STORM, width=DT, align="edge", color="lightsteelblue", label="rain")
    a1.bar(tb, pe0, width=DT, align="edge", color="tab:blue", label="effective rain, CN 70")
    a1.invert_yaxis()
    a1.set_ylabel("rain per 0.5 h (mm)")
    a1.legend(fontsize=8)
    uh = triangular_uh(AREA, TC_PRE)
    for k, p in enumerate(pe0):
        a2.plot(np.arange(len(uh)) * DT + k * DT, p * uh, color="0.7", lw=0.8)
    a2.plot(t, q0, color="tab:blue", lw=2, label=f"sum of bursts: peak {pk0:.1f} m3/s at {tp0:.1f} h")
    a2.set_xlabel("time since the storm began (h)")
    a2.set_ylabel("river flow (m3/s)")
    a2.legend()
    a1.set_title("Each burst makes a small flood; the river adds them")
    for a in (a1, a2):
        a.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "convolution.png", dpi=120)

    H = 2.0
    Ap = size_pond(q1, DT, pk0, H)
    tp, qo, hh, left = route_pond(q1, DT, Ap, H, pk0)
    print(f"pond: area {Ap:,.0f} m2, depth {H} m, volume {Ap * H:,.0f} m3, outflow peak {qo.max():.1f} m3/s")
    fig2, ax = plt.subplots(figsize=(6.4, 4))
    ax.plot(t, q0, color="tab:blue", label=f"before: {pk0:.1f} m3/s")
    ax.plot(t1, q1, color="tab:red", label=f"after urbanisation: {pk1:.1f} m3/s")
    ax.plot(tp, qo, color="tab:green", lw=2, label=f"after, with pond: {qo.max():.1f} m3/s")
    ax.set_xlim(0, 12)
    ax.set_xlabel("time since the storm began (h)")
    ax.set_ylabel("river flow (m3/s)")
    ax.set_title("Urbanisation raises the peak; a detention pond brings it back")
    ax.grid(alpha=0.3)
    ax.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "urban_and_pond.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change the storm in `STORM` so the heaviest burst comes first, run `flood`, and compare the peak and the time to peak with the original.
2. **Stretch.** Write `uh_peak_check(A, Tc)` that returns the peak of `triangular_uh(..., normalize=False)` divided by the formula 0.208 A / Tp and explain why it is slightly below 1.
3. **Explain.** Explain, using the water balance Ap dh/dt = I - O, why the pond outflow peak occurs where the outflow meets the falling inflow.

## Assessment
- **A. Conceptual.** Why does the flood hydrograph equal the sum of shifted copies of the unit hydrograph, and what two assumptions make that valid?
- **B. Numerical.** A catchment has A = 8 km2 and Tc = 1.5 h; the burst length is D = 0.5 h. Find Tp and the UH peak per mm. Check the 0.208 factor.
- **C. Graph and simulation interpretation.** In the urbanisation figure, by how much do the peak and its time change, and which of the two changes (CN or Tc) matters more for the peak?
- **D. Design and optimisation challenge.** Design: you can build the pond 1 m, 2 m or 3 m deep, with the orifice always passing 32.3 m3/s at the full depth. How does the required volume change, and what does a 1 m deep pond need in surface area? What trade-offs would you weigh?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    A, dt = L.AREA, L.DT
    # --- unit hydrograph
    Tp = 0.25 + 0.6 * 2.0
    Qp = 0.208 * A / Tp
    checks.append((f"worked example: UH for A = 12 km2, Tc = 2 h, D = 0.5 h has Tp = 1.45 h and Qp = 1.72 m3/s (got {Tp:.2f}, {Qp:.2f})",
                   abs(Tp - 1.45) < 1e-12 and abs(Qp - 1.72) < 0.005))
    raw = L.triangular_uh(A, 2.0, normalize=False)
    checks.append((f"the unit hydrograph holds 1 mm: raw sampled volume {L.uh_volume_mm(raw, A):.4f} mm (within 0.5 percent)",
                   abs(L.uh_volume_mm(raw, A) - 1.0) < 0.005))
    u = L.triangular_uh(A, 2.0)
    checks.append(("normalised UH holds exactly 1 mm = 12 000 m3 over 12 km2",
                   abs(L.uh_volume_mm(u, A) - 1.0) < 1e-12 and abs(u.sum() * dt * 3600 - 12000.0) < 1e-6))
    checks.append(("0.208 follows from the triangle: 1000 / (0.5 * 2.67 * 3600) = 0.208",
                   abs(1000.0 / (0.5 * 2.67 * 3600.0) - 0.208) < 5e-4))
    # --- excess rain and convolution
    t0, q0, pe0 = L.flood(L.STORM, L.CN_PRE, L.TC_PRE)
    t1, q1, pe1 = L.flood(L.STORM, L.CN_POST, L.TC_POST)
    checks.append(("storm of 90 mm: effective rain is 26.3 mm at CN 70 and 63.2 mm at CN 90",
                   abs(L.STORM.sum() - 90.0) < 1e-12 and abs(pe0.sum() - 26.3) < 0.05 and abs(pe1.sum() - 63.2) < 0.05))
    checks.append(("at CN 70 the first hour of rain (12 mm) is all absorbed (Ia = 21.8 mm)",
                   pe0[0] == 0.0 and pe0[1] == 0.0 and abs(0.2 * (25400 / 70 - 254) - 21.77) < 0.01))
    checks.append(("effective rain never exceeds rain and is never negative",
                   bool(np.all(pe0 >= 0) and np.all(pe0 <= L.STORM + 1e-12) and np.all(pe1 >= 0))))
    qm = L.convolve_manual(pe0, L.triangular_uh(A, L.TC_PRE))
    checks.append(("convolution by two loops equals np.convolve", np.allclose(qm, q0, atol=1e-9)))
    checks.append(("hydrograph volume equals effective rain times area (mass kept)",
                   abs(q0.sum() * dt * 3600 - pe0.sum() * A * 1000) < 1e-6 * pe0.sum() * A * 1000
                   and abs(q1.sum() * dt * 3600 - pe1.sum() * A * 1000) < 1e-6 * pe1.sum() * A * 1000))
    pk0, tp0 = L.peak_and_time(t0, q0)
    pk1, tp1 = L.peak_and_time(t1, q1)
    checks.append((f"before urbanisation: peak 32.3 m3/s at 4.0 h (got {pk0:.1f} at {tp0:.1f})",
                   abs(pk0 - 32.3) < 0.05 and abs(tp0 - 4.0) < 1e-9))
    checks.append((f"after urbanisation: peak 94.6 m3/s at 3.0 h, 2.9 times higher and 1 h earlier (got {pk1:.1f} at {tp1:.1f}, ratio {pk1 / pk0:.2f})",
                   abs(pk1 - 94.6) < 0.05 and abs(tp1 - 3.0) < 1e-9 and abs(pk1 / pk0 - 2.93) < 0.01))
    pkc = L.peak_and_time(*L.flood(L.STORM, L.CN_POST, L.TC_PRE)[:2])
    pkt = L.peak_and_time(*L.flood(L.STORM, L.CN_PRE, L.TC_POST)[:2])
    checks.append((f"CN alone gives 76.0 m3/s at 3.5 h; a faster Tc alone gives 39.3 m3/s at 3.0 h (got {pkc[0]:.1f} at {pkc[1]:.1f}; {pkt[0]:.1f} at {pkt[1]:.1f})",
                   abs(pkc[0] - 76.0) < 0.05 and abs(pkc[1] - 3.5) < 1e-9 and abs(pkt[0] - 39.3) < 0.05 and abs(pkt[1] - 3.0) < 1e-9))
    checks.append(("lag to peak shortens from Tp = 1.45 h to 0.97 h when Tc falls from 2.0 to 1.2 h",
                   abs((0.25 + 0.6 * 2.0) - 1.45) < 1e-12 and abs((0.25 + 0.6 * 1.2) - 0.97) < 1e-12))
    # --- limits and failure cases
    cn3 = 23 * 70 / (10 + 0.13 * 70)
    p3 = L.flood(L.STORM, cn3, L.TC_PRE)
    checks.append((f"saturated soil: CN 70 becomes about 84.3, effective rain 50.7 mm and peak 61.4 m3/s, 1.9 times dry (got {cn3:.1f}, {p3[2].sum():.1f}, {p3[1].max():.1f})",
                   abs(cn3 - 84.3) < 0.05 and abs(p3[2].sum() - 50.7) < 0.05 and abs(p3[1].max() - 61.4) < 0.05 and abs(p3[1].max() / pk0 - 1.90) < 0.01))
    window = max(L.STORM[k:k + 4].sum() for k in range(5))
    rational = (pe0.sum() / L.STORM.sum()) * (window / 2.0) * A / 3.6
    checks.append((f"rational cross-check (C = 0.292, i = 34 mm/h over Tc = 2 h): 33.1 m3/s, within 3 percent of the UH peak (got {rational:.1f})",
                   window == 68.0 and abs(rational - 33.1) < 0.05 and abs(rational - pk0) / pk0 < 0.03))
    # --- Muskingum
    padded = np.concatenate([q0, np.zeros(30)])
    out, (c0, c1, c2) = L.muskingum(padded, 1.5, 0.1)
    checks.append(("Muskingum K = 1.5 h, X = 0.1: C0, C1, C2 = 0.0625, 0.25, 0.6875, and they add to 1",
                   np.allclose([c0, c1, c2], [0.0625, 0.25, 0.6875], atol=1e-4) and abs(c0 + c1 + c2 - 1) < 1e-12))
    checks.append((f"routing flattens and delays: peak 32.3 falls to 23.7 m3/s and moves from 4.0 h to 5.0 h (got {out.max():.1f} at {out.argmax() * dt:.1f})",
                   abs(out.max() - 23.7) < 0.05 and abs(out.argmax() * dt - 5.0) < 1e-9 and out.max() < padded.max()))
    checks.append(("routing keeps the water (volume within 0.01 percent) and stays non-negative",
                   abs(out.sum() - padded.sum()) / padded.sum() < 1e-4 and out.min() >= -1e-12))
    pulse = np.array([0, 0, 5, 10, 5, 0, 0, 0.0])
    o2, cc = L.muskingum(pulse, 1.0, 0.5, dt=1.0)
    checks.append(("limit X = 0.5 with dt = K: pure delay by one step, no flattening",
                   np.allclose(cc, [0.0, 1.0, 0.0]) and np.allclose(o2[1:], pulse[:-1])))
    # --- pond
    H = 2.0
    Ap = L.size_pond(q1, dt, pk0, H)
    tp, qo, hh, left = L.route_pond(q1, dt, Ap, H, pk0)
    checks.append((f"worked example: pond for the urbanised flood needs about 454 thousand m3 (226 857 m2 at 2 m); got {Ap * H:,.0f} m3",
                   abs(Ap * H - 453714) < 200 and abs(Ap - 226857) < 100))
    checks.append((f"with that pond the outflow peak equals the old peak, 32.3 m3/s, and the stage reaches 2.0 m (got {qo.max():.1f}, {hh.max():.2f})",
                   abs(qo.max() - pk0) < 0.05 and abs(hh.max() - H) < 1e-3))
    checks.append((f"pond peak outflow comes at 4.5 h, 1.5 h after the inflow peak (got {tp[qo.argmax()]:.1f} h)",
                   abs(tp[qo.argmax()] - 4.5) < 1e-9))
    vin = q1.sum() * dt * 3600
    vout = (qo.sum() - 0.5 * (qo[0] + qo[-1])) * dt * 3600
    checks.append(("water balance of the pond: inflow = outflow + water left (within 0.1 percent)",
                   abs(vin - vout - left) / vin < 1e-3))
    h_half = L.route_pond(q1, dt, 0.5 * Ap, H, pk0)
    checks.append((f"half the volume overtops: stage 3.31 m and outflow peak 41.5 m3/s (got {h_half[2].max():.2f}, {h_half[1].max():.1f})",
                   abs(h_half[2].max() - 3.31) < 0.01 and abs(h_half[1].max() - 41.5) < 0.1 and h_half[2].max() > H))
    ap1 = L.size_pond(q1, dt, pk0, 1.0)
    ap3 = L.size_pond(q1, dt, pk0, 3.0)
    checks.append(("the volume does not depend on the chosen depth (1 m, 2 m, 3 m give the same volume within 0.1 percent)",
                   abs(ap1 * 1.0 - Ap * H) / (Ap * H) < 1e-3 and abs(ap3 * 3.0 - Ap * H) / (Ap * H) < 1e-3))
    extra = (pe1.sum() - pe0.sum()) * A * 1000
    checks.append((f"the extra run-off of urbanisation is 443 thousand m3; the pond (454 thousand) is slightly larger (got {extra:,.0f})",
                   abs(extra - 443052) < 100 and Ap * H > extra))
    # drain test against the analytic solution sqrt(h) = sqrt(h0) - c t / (2 Ap)
    Ap2, h0 = 1.0e5, 1.5
    c = pk0 / math.sqrt(H)
    r = L.route_pond(np.zeros(3), dt, Ap2, H, pk0, t_end_h=3.0, h0=h0)
    ex = (math.sqrt(h0) - c * r[0][2] * 3600 / (2 * Ap2)) ** 2
    checks.append((f"pond emptying with no inflow matches the exact formula (got {r[2][2]:.4f} m, exact {ex:.4f} m)",
                   abs(r[2][2] - ex) < 1e-6))
    qa = 0.208 * 8.0 / (0.25 + 0.6 * 1.5)
    checks.append((f"UH for A = 8 km2, Tc = 1.5 h: Tp = 1.15 h and Qp = 1.45 m3/s (got {qa:.2f})", abs(qa - 1.447) < 0.001))
    checks.append((f"a 1 m deep pond needs about 454 thousand m2 (45 ha), got {ap1:,.0f} m2", abs(ap1 - 453714) < 500))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
